# TrOCR fine-tuning — Kaggle (deterministic single-GPU run)

Fine-tunes `microsoft/trocr-base-handwritten` against the versioned unified manifest with validated profiles from 5k through 200k samples.

Before running: attach a GPU, enable Internet for the pinned base-model download, add one validated dataset, and set `DATA_ROOT` to the exact folder containing `manifest.csv`, `dataset-validation.json`, and `train/`, `val/`, `test/`. Set a unique `RUN_TAG` for a new experiment; rerunning the same tag/dataset/config resumes its epoch-boundary checkpoint. The notebook never auto-selects a dataset. Model development uses validation only; the locked final test is read exactly once after choices are frozen.

In [ ]:
# 1. Dependencies — runtime versions are recorded in the run contract/report.
# If you install packages, pin them here and restart the kernel before training.
# Example: !pip install -q 'transformers==4.57.1' 'pandas==2.2.3' 'Pillow==11.3.0'
print('[OK] dependency cell ready')

In [ ]:
# 2. Imports
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath, PureWindowsPath

# Must be set before the first CUDA context is initialized.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import PIL
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import transformers
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel,
                          get_linear_schedule_with_warmup)
print('[OK] imports')

In [ ]:
# 3. CUDA and deterministic-runtime gate
if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires a CUDA GPU because CUDA AMP is enabled.')
n_gpu = torch.cuda.device_count()
print('GPU count      :', n_gpu)
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')
device = torch.device('cuda:0')
torch.cuda.set_device(device)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
print('Training device :', device)
if n_gpu > 1:
    print('Additional GPUs are intentionally unused. This notebook uses one GPU; use DDP in a launcher for multi-GPU training.')

In [ ]:
# 4. Explicit configuration and full pre-training dataset validation
MODEL_NAME = 'microsoft/trocr-base-handwritten'
MODEL_REVISION = 'eaacaf452b06415df8f10bb6fad3a4c11e609406'
RUN_TAG = 'experiment-001'  # Change this for a genuinely new experiment.
RUN_SEED = 20260812
RESUME_IF_AVAILABLE = True
RUNS_ROOT = Path('/kaggle/working/trocr-runs')
INCLUDE_PRIVATE_EVALUATION_DATA = False

# REQUIRED: exact attached dataset directory. Never leave this blank.
# Example: '/kaggle/input/my-upload/dataset_014'
DATA_ROOT = ''

if not DATA_ROOT.strip():
    candidates = sorted(
        str(p.parent) for p in Path('/kaggle/input').rglob('manifest.csv'))
    print('Manifest candidates found under /kaggle/input:')
    for candidate in candidates:
        print('  ', candidate)
    raise RuntimeError('Set DATA_ROOT explicitly to exactly one validated dataset folder.')
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve(strict=True)
if not DATA_ROOT.is_dir():
    raise NotADirectoryError(f'DATA_ROOT is not a directory: {DATA_ROOT}')
print('DATA_ROOT =', DATA_ROOT)

MANIFEST_CSV = DATA_ROOT / 'manifest.csv'
VALIDATION_JSON = DATA_ROOT / 'dataset-validation.json'
RUN_METADATA_JSON = DATA_ROOT / 'run-metadata.json'
EVALUATION_ANNOTATIONS_CSV = DATA_ROOT / 'evaluation-annotations.csv'
EXPECTED_PREPROCESSING_ID = 'aspect-pad-384-v1'
SPLITS = ('train', 'val', 'test')
SPLIT_DIRS = {split: DATA_ROOT / split for split in SPLITS}
MANIFEST_SCHEMA_VERSION = '1'
MANIFEST_COLUMNS = (
    'filename', 'label', 'split', 'source', 'field_type', 'font',
    'sample_mode', 'writer_id', 'schema_version',
)
ALLOWED_SOURCES = {'synthetic', 'real'}
EVALUATION_ANNOTATION_COLUMNS = (
    'filename', 'split', 'evaluation_condition', 'format_profile',
    'format_id', 'label_seen_in_train', 'font_seen_in_train',
    'format_seen_in_train',
)
LEGACY_NON_DATE_FORMAT_ID_ALLOWLIST = {
    'full_name': {'name_mixed_representative_v1'},
    'age': {'default'}, 'cause_of_death': {'default'},
    'character': {'default'}, 'citizenship': {'default'},
    'civil_status': {'default'}, 'numeric': {'default'},
    'occupation': {'default'}, 'place': {'default'},
    'religion': {'default'}, 'sex': {'default'},
}

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        while chunk := source.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def sha256_json(value):
    return hashlib.sha256(canonical_json(value).encode('utf-8')).hexdigest()

def atomic_write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as output:
        json.dump(value, output, indent=2, sort_keys=True)
        output.write('\n')
        output.flush()
        os.fsync(output.fileno())
    os.replace(temporary, path)

def hash_named_files(named_paths, preserve_order=False):
    digest = hashlib.sha256()
    ordered_paths = (named_paths if preserve_order else
                     sorted(named_paths, key=lambda item: item[0]))
    for logical_name, path in ordered_paths:
        encoded = logical_name.replace('\\', '/').encode('utf-8')
        digest.update(len(encoded).to_bytes(8, 'big'))
        digest.update(encoded)
        with Path(path).open('rb') as source:
            while chunk := source.read(1024 * 1024):
                digest.update(chunk)
    return digest.hexdigest()

def hash_dataset_image_paths(named_paths, split_names=('train', 'val', 'test')):
    # Match src.provenance.hash_dataset_images exactly: configured split
    # order first, then binary/logical-name order within each split.
    split_rank = {split: index for index, split in enumerate(split_names)}
    def dataset_sort_key(item):
        logical_name = item[0].replace('\\', '/')
        split = logical_name.partition('/')[0]
        return split_rank.get(split, len(split_rank)), logical_name
    ordered_paths = sorted(named_paths, key=dataset_sort_key)
    return hash_named_files(ordered_paths, preserve_order=True)

def hash_directory(directory):
    directory = Path(directory)
    return hash_named_files([
        (path.relative_to(directory).as_posix(), path)
        for path in directory.rglob('*') if path.is_file()
    ])

def dependency_versions():
    return {
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'pillow': PIL.__version__,
        'cuda_runtime': torch.version.cuda,
        'cudnn': str(torch.backends.cudnn.version()),
        'gpu_name': torch.cuda.get_device_name(0),
        'gpu_compute_capability': '.'.join(map(
            str, torch.cuda.get_device_capability(0))),
        'gpu_count_visible': int(torch.cuda.device_count()),
    }

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_data_loader_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def is_leaf_png(filename):
    if not filename or filename != filename.strip() or '/' in filename or '\\' in filename:
        return False
    posix, windows = PurePosixPath(filename), PureWindowsPath(filename)
    return (filename not in {'.', '..'} and not posix.is_absolute()
            and not windows.is_absolute() and not windows.drive
            and Path(filename).suffix.lower() == '.png')

if not MANIFEST_CSV.is_file():
    raise FileNotFoundError(f'Missing unified manifest: {MANIFEST_CSV}')
if not VALIDATION_JSON.is_file():
    raise FileNotFoundError(
        f'Missing {VALIDATION_JSON.name}; publish/validate the dataset before training.')
if not RUN_METADATA_JSON.is_file():
    raise FileNotFoundError(f'Missing generator provenance: {RUN_METADATA_JSON}')
if not EVALUATION_ANNOTATIONS_CSV.is_file():
    raise FileNotFoundError(
        f'Missing synthetic evaluation sidecar: {EVALUATION_ANNOTATIONS_CSV}')
for split, split_dir in SPLIT_DIRS.items():
    if not split_dir.is_dir():
        raise FileNotFoundError(f'Missing split directory: {split_dir}')

manifest_df = pd.read_csv(MANIFEST_CSV, dtype=str, keep_default_na=False)
if tuple(manifest_df.columns) != MANIFEST_COLUMNS:
    raise ValueError(
        f'Unified manifest columns must be exactly {MANIFEST_COLUMNS}; '
        f'found {tuple(manifest_df.columns)}')
if manifest_df.empty:
    raise ValueError('Unified manifest contains no rows.')

errors = []
expected_paths = set()
seen_filenames = set()
named_image_paths = []
for row_number, row in enumerate(manifest_df.to_dict('records'), start=2):
    filename = row['filename']
    label = row['label']
    split = row['split']
    source = row['source']
    writer_id = row['writer_id']
    prefix = f'row {row_number}'
    if not is_leaf_png(filename):
        errors.append(f'{prefix}: unsafe/non-PNG filename {filename!r}')
        continue
    filename_key = filename.casefold()
    if filename_key in seen_filenames:
        errors.append(f'{prefix}: duplicate filename {filename!r}')
    seen_filenames.add(filename_key)
    if split not in SPLITS:
        errors.append(f'{prefix}: invalid split {split!r}')
        continue
    if source not in ALLOWED_SOURCES:
        errors.append(f'{prefix}: invalid source {source!r}')
    if row['schema_version'] != MANIFEST_SCHEMA_VERSION:
        errors.append(f'{prefix}: unsupported schema_version {row["schema_version"]!r}')
    if not label.strip() or label.strip().upper() == 'UNREADABLE':
        errors.append(f'{prefix}: label is empty or UNREADABLE')
    if source == 'real' and not writer_id.strip():
        errors.append(f'{prefix}: real row requires writer_id')
    if source == 'synthetic' and writer_id.strip():
        errors.append(f'{prefix}: synthetic row must have empty writer_id')
    if source == 'synthetic' and any(not row[name].strip() for name in
                                     ('field_type', 'font', 'sample_mode')):
        errors.append(f'{prefix}: synthetic row lacks field_type/font/sample_mode')
    split_root = SPLIT_DIRS[split].resolve(strict=True)
    image_path = split_root / filename
    try:
        resolved = image_path.resolve(strict=True)
        if resolved.parent != split_root or not resolved.is_file():
            raise ValueError('not a direct-child regular file')
        with Image.open(resolved) as opened:
            opened.verify()
        with Image.open(resolved) as opened:
            if opened.width <= 1 or opened.height <= 1:
                raise ValueError(f'invalid dimensions {opened.size}')
    except (OSError, ValueError, UnidentifiedImageError) as exc:
        errors.append(f'{prefix}: unreadable/unsafe image {split}/{filename}: {exc}')
        continue
    key = (split, filename_key)
    if key in expected_paths:
        errors.append(f'{prefix}: duplicate split path {split}/{filename}')
    expected_paths.add(key)
    named_image_paths.append((f'{split}/{filename}', resolved))

actual_paths = set()
for split, split_dir in SPLIT_DIRS.items():
    for path in split_dir.iterdir():
        if not path.is_file() or path.suffix.lower() != '.png':
            errors.append(f'unexpected entry in {split}: {path.name}')
        else:
            actual_paths.add((split, path.name.casefold()))
for split in SPLITS:
    if not (manifest_df['split'] == split).any():
        errors.append(f'split is empty: {split}')
if expected_paths != actual_paths:
    for split, filename in sorted(expected_paths - actual_paths):
        errors.append(f'manifest image missing: {split}/{filename}')
    for split, filename in sorted(actual_paths - expected_paths):
        errors.append(f'orphan image not represented in manifest: {split}/{filename}')

real_writers = {
    split: set(manifest_df.loc[(manifest_df['source'] == 'real')
                               & (manifest_df['split'] == split), 'writer_id'])
    for split in SPLITS
}
for left_index, left in enumerate(SPLITS):
    for right in SPLITS[left_index + 1:]:
        overlap = real_writers[left] & real_writers[right]
        if overlap:
            errors.append(
                f'real writers cross {left}/{right}: {sorted(overlap)[:10]}')
REAL_WRITER_HELD_OUT_AVAILABLE = bool(real_writers['test'])
if not REAL_WRITER_HELD_OUT_AVAILABLE:
    print('!' * 72)
    print('WARNING: NO REAL WRITER-HELD-OUT TEST DATA IS AVAILABLE.')
    print('Results are synthetic diagnostics, not real-registry performance.')
    print('!' * 72)

with VALIDATION_JSON.open('r', encoding='utf-8') as source:
    dataset_validation = json.load(source)
if dataset_validation.get('schema_version') != 1:
    errors.append('dataset-validation.json has unsupported schema_version')
if dataset_validation.get('valid') is not True:
    errors.append('dataset-validation.json does not report valid=true')
if dataset_validation.get('errors'):
    errors.append('dataset-validation.json contains validation errors')
manifest_sha256 = sha256_file(MANIFEST_CSV)
images_sha256 = hash_dataset_image_paths(named_image_paths, SPLITS)
if dataset_validation.get('manifest_sha256') != manifest_sha256:
    errors.append('dataset-validation.json is stale: manifest hash mismatch')
if dataset_validation.get('images_sha256') != images_sha256:
    errors.append('dataset-validation.json is stale: image-set hash mismatch')
statistics = dataset_validation.get('statistics', {})
if statistics.get('manifest_rows') != len(manifest_df):
    errors.append('dataset-validation.json row count does not match manifest')
with RUN_METADATA_JSON.open('r', encoding='utf-8') as source:
    generator_run_metadata = json.load(source)
if not isinstance(generator_run_metadata, dict):
    errors.append('run-metadata.json must contain a JSON object')
    generator_run_metadata = {}
if generator_run_metadata.get('metadata_schema_version') not in {1, 2}:
    errors.append('run-metadata.json has an unsupported metadata schema')
if str(generator_run_metadata.get('manifest_schema_version')) != MANIFEST_SCHEMA_VERSION:
    errors.append('run-metadata.json has an unsupported manifest schema')
if generator_run_metadata.get('manifest_sha256') != manifest_sha256:
    errors.append('run-metadata.json is stale: manifest hash mismatch')
if generator_run_metadata.get('images_sha256') != images_sha256:
    errors.append('run-metadata.json is stale: image-set hash mismatch')
if generator_run_metadata.get('row_count') != len(manifest_df):
    errors.append('run-metadata.json row count does not match manifest')
if generator_run_metadata.get('evaluation_annotations') != EVALUATION_ANNOTATIONS_CSV.name:
    errors.append('run-metadata.json does not identify evaluation-annotations.csv')
evaluation_annotations_sha256 = sha256_file(EVALUATION_ANNOTATIONS_CSV)
if generator_run_metadata.get('evaluation_annotations_sha256') != evaluation_annotations_sha256:
    errors.append('run-metadata.json is stale: evaluation annotation hash mismatch')
generator_preprocessing = generator_run_metadata.get('preprocessing')
if not isinstance(generator_preprocessing, dict):
    errors.append('run-metadata.json lacks preprocessing provenance')
    generator_preprocessing = {}
if generator_preprocessing.get('id') != EXPECTED_PREPROCESSING_ID:
    errors.append('generator preprocessing does not match the notebook transform')
evaluation_policy = generator_run_metadata.get('evaluation_policy')
if not isinstance(evaluation_policy, dict):
    errors.append('run-metadata.json lacks an evaluation_policy object')
    evaluation_policy = {}
if evaluation_policy.get('policy_version') != '1':
    errors.append('run-metadata.json has an unsupported evaluation policy')

def allowed_evaluation_format_ids(field_type, expected_profile, pattern_ids):
    configured = pattern_ids.get(field_type, {}).get(expected_profile, [])
    if configured:
        return set(configured)
    if expected_profile == 'base':
        return set(LEGACY_NON_DATE_FORMAT_ID_ALLOWLIST.get(field_type, set()))
    return set()

annotations_df = pd.read_csv(
    EVALUATION_ANNOTATIONS_CSV, dtype=str, keep_default_na=False)
if tuple(annotations_df.columns) != EVALUATION_ANNOTATION_COLUMNS:
    errors.append(
        'evaluation-annotations.csv columns do not match the version-1 contract')
else:
    annotation_keys = set()
    annotation_casefold_keys = set()
    manifest_lookup = {
        (row['split'], row['filename']): row
        for row in manifest_df.to_dict('records')
    }
    synthetic_train_labels = set(manifest_df.loc[
        (manifest_df['source'] == 'synthetic')
        & (manifest_df['split'] == 'train'), 'label'])
    synthetic_train_fonts = set(manifest_df.loc[
        (manifest_df['source'] == 'synthetic')
        & (manifest_df['split'] == 'train'), 'font'].str.casefold()) - {''}
    policy_conditions = evaluation_policy.get('evaluation_conditions', {})
    policy_test_by_field = evaluation_policy.get(
        'test_evaluation_conditions_by_field', {})
    policy_format = evaluation_policy.get('format_holdout', {})
    policy_format_fields = set(policy_format.get('fields', []))
    policy_format_profiles = policy_format.get('profiles', {})
    policy_pattern_ids = policy_format.get('pattern_ids', {})
    for row_number, row in enumerate(annotations_df.to_dict('records'), start=2):
        filename, split = row['filename'], row['split']
        key = (split, filename)
        folded_key = (split, filename.casefold())
        if not is_leaf_png(filename) or split not in SPLITS:
            errors.append(f'evaluation annotation row {row_number}: unsafe filename/split')
            continue
        if folded_key in annotation_casefold_keys:
            errors.append(f'evaluation annotation row {row_number}: duplicate key {key}')
        annotation_casefold_keys.add(folded_key)
        annotation_keys.add(key)
        manifest_row = manifest_lookup.get(key)
        if manifest_row is None or manifest_row['source'] != 'synthetic':
            errors.append(
                f'evaluation annotation row {row_number}: no matching synthetic row')
            continue
        for boolean_column in (
                'label_seen_in_train', 'font_seen_in_train', 'format_seen_in_train'):
            if row[boolean_column].casefold() not in {'true', 'false'}:
                errors.append(
                    f'evaluation annotation row {row_number}: invalid {boolean_column}')
        expected_condition = (
            policy_test_by_field.get(manifest_row['field_type'])
            if split == 'test' else None) or policy_conditions.get(split)
        if not expected_condition or row['evaluation_condition'] != expected_condition:
            errors.append(
                f'evaluation annotation row {row_number}: condition violates policy')
        field_type = manifest_row['field_type']
        expected_profile = (
            policy_format_profiles.get(split, {}).get(field_type, 'base')
            if field_type in policy_format_fields else 'base')
        if row['format_profile'] != expected_profile:
            errors.append(
                f'evaluation annotation row {row_number}: format profile violates policy')
        allowed_format_ids = allowed_evaluation_format_ids(
            field_type, expected_profile, policy_pattern_ids)
        if row['format_id'] not in allowed_format_ids:
            errors.append(
                f'evaluation annotation row {row_number}: format id violates policy')
        expected_seen = {
            'label_seen_in_train': manifest_row['label'] in synthetic_train_labels,
            'font_seen_in_train': (
                bool(manifest_row['font'])
                and manifest_row['font'].casefold() in synthetic_train_fonts),
            'format_seen_in_train': expected_profile == 'base',
        }
        for column, expected_value in expected_seen.items():
            if row[column].casefold() in {'true', 'false'} and (
                    (row[column].casefold() == 'true') != expected_value):
                errors.append(
                    f'evaluation annotation row {row_number}: stale {column}')
    expected_annotation_keys = {
        (row['split'], row['filename'])
        for row in manifest_df.loc[
            manifest_df['source'] == 'synthetic', ['split', 'filename']
        ].to_dict('records')
    }
    for split, filename in sorted(expected_annotation_keys - annotation_keys):
        errors.append(f'missing synthetic evaluation annotation: {split}/{filename}')
    for split, filename in sorted(annotation_keys - expected_annotation_keys):
        errors.append(f'orphan synthetic evaluation annotation: {split}/{filename}')
if errors:
    preview = '\n'.join(f'  - {message}' for message in errors[:30])
    raise ValueError(f'Dataset validation failed ({len(errors)} errors):\n{preview}')

# Join generator-authored synthetic domain annotations; derive only real-writer tags.
manifest_df = manifest_df.merge(
    annotations_df, on=['filename', 'split'], how='left', validate='one_to_one')
for column in EVALUATION_ANNOTATION_COLUMNS[2:]:
    manifest_df[column] = manifest_df[column].fillna('').astype(str)
train_labels = set(manifest_df.loc[
    manifest_df['split'] == 'train', 'label'].str.strip().str.casefold())

def annotation_bool(value):
    if str(value).casefold() not in {'true', 'false'}:
        raise ValueError(f'Invalid evaluation annotation boolean: {value!r}')
    return str(value).casefold() == 'true'

def evaluation_tags(row):
    if row['source'] == 'synthetic':
        condition = row['evaluation_condition']
        label_seen = annotation_bool(row['label_seen_in_train'])
        payload = condition.removeprefix('synthetic_')
        conditions = (['in_distribution'] if payload in {
            'training', 'in_distribution'} else payload.split('+'))
        domain = (
            'synthetic / in-distribution'
            if payload in {'training', 'in_distribution'}
            else 'synthetic / held-out')
    else:
        label_seen = row['label'].strip().casefold() in train_labels
        if row['split'] == 'train':
            condition = 'real_training'
            conditions = ['in_distribution']
            domain = 'real / training'
        else:
            condition = 'real_writer_held_out'
            conditions = ['writer_held_out']
            domain = 'real / writer-held-out'
    label_status = 'seen_label' if label_seen else 'unseen_label'
    if not label_seen:
        conditions.append('label_held_out')
    return condition, label_status, tuple(conditions), domain

tags = manifest_df.apply(evaluation_tags, axis=1)
manifest_df['evaluation_condition'] = [item[0] for item in tags]
manifest_df['seen_unseen'] = [item[1] for item in tags]
manifest_df['held_out_conditions'] = [item[2] for item in tags]
manifest_df['evaluation_domain'] = [item[3] for item in tags]
train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)
print('Validated manifest rows:', len(manifest_df))
print(pd.crosstab(manifest_df['split'], manifest_df['source']))
print('Locked real test writers:', len(real_writers['test']))

PROFILES = {
    '5k': dict(EPOCHS=4, BATCH_SIZE=16, GRAD_ACCUM_STEPS=2,
               DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.10,
               FREEZE_ENCODER=True, UNFREEZE_AT_EPOCH=1,
               EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '10k': dict(EPOCHS=3, BATCH_SIZE=16, GRAD_ACCUM_STEPS=4,
                DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '20k': dict(EPOCHS=2, BATCH_SIZE=16, GRAD_ACCUM_STEPS=6,
                DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    '50k': dict(EPOCHS=2, BATCH_SIZE=16, GRAD_ACCUM_STEPS=8,
                DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.03,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    # Large-run presets target one 16 GiB Kaggle T4. A micro-batch of 8
    # leaves headroom while accumulation keeps the effective batch at 128.
    '80k': dict(EPOCHS=3, BATCH_SIZE=8, GRAD_ACCUM_STEPS=16,
                DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.03,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    '100k': dict(EPOCHS=3, BATCH_SIZE=8, GRAD_ACCUM_STEPS=16,
                 DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.03,
                 FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                 EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    '150k': dict(EPOCHS=2, BATCH_SIZE=8, GRAD_ACCUM_STEPS=16,
                 DECODER_LR=1.5e-5, ENCODER_LR=7.5e-6, WARMUP_RATIO=0.03,
                 FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                 EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    '200k': dict(EPOCHS=2, BATCH_SIZE=8, GRAD_ACCUM_STEPS=16,
                 DECODER_LR=1.5e-5, ENCODER_LR=7.5e-6, WARMUP_RATIO=0.03,
                 FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                 EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
}
PROFILE_SAMPLE_COUNTS = {
    '5k': 5_000, '10k': 10_000, '20k': 20_000, '50k': 50_000,
    '80k': 80_000, '100k': 100_000, '150k': 150_000, '200k': 200_000,
}
PROFILE = '100k'  # dataset_026 contains 100,000 rows
if PROFILE not in PROFILES:
    raise ValueError(f'Unknown PROFILE {PROFILE!r}; choose one of {sorted(PROFILES)}')
cfg = dict(PROFILES[PROFILE])
expected_profile_rows = PROFILE_SAMPLE_COUNTS[PROFILE]
if len(manifest_df) != expected_profile_rows:
    print(f'WARNING: profile {PROFILE!r} targets {expected_profile_rows:,} rows, '
          f'but the manifest contains {len(manifest_df):,}. Choose the nearest profile.')
TRAINING_CONTRACT_ID = 'trocr-manual-single-gpu-v2'
TRAINING_CONTRACT = {
    'id': TRAINING_CONTRACT_ID,
    'optimizer': {
        'name': 'torch.optim.AdamW', 'weight_decay': 1e-4,
        'betas': [0.9, 0.999], 'eps': 1e-8,
    },
    'scheduler': {'name': 'linear_with_warmup'},
    'gradient_clip_max_norm': 1.0,
    'amp': {'device_type': 'cuda', 'dtype': 'float16'},
    'accumulation': 'sample-weighted-partial-group-v1',
}
NUM_WORKERS = 0  # Avoid Kaggle multiprocessing teardown hangs; opt-in workers are deterministically seeded.
DEPENDENCY_VERSIONS = dependency_versions()
DETERMINISM_GUARANTEE = (
    'Exact resume/repeat is targeted on the same GPU architecture, CUDA, Torch, '
    'Transformers, and dependency versions recorded in this run. Other hardware '
    'may differ at floating-point precision.'
)
DATASET_PROVENANCE = {
    'dataset_name': DATA_ROOT.name,
    'manifest_schema_version': int(MANIFEST_SCHEMA_VERSION),
    'manifest_sha256': manifest_sha256,
    'images_sha256': images_sha256,
    'dataset_validation_sha256': sha256_file(VALIDATION_JSON),
    'validated_at': dataset_validation.get('validated_at'),
    'manifest_rows': int(len(manifest_df)),
    'run_metadata_sha256': sha256_file(RUN_METADATA_JSON),
    'evaluation_annotations_sha256': evaluation_annotations_sha256,
    'generator_revision': generator_run_metadata.get('generator_revision'),
    'generator_seed': generator_run_metadata.get('seed'),
    'generator_preprocessing': generator_preprocessing,
    'evaluation_policy_sha256': sha256_json(evaluation_policy),
}
SPLIT_DEFINITION = {
    'policy': 'manifest-defined; validation and development only for selection; locked test one-shot',
    'counts': {split: int((manifest_df['split'] == split).sum()) for split in SPLITS},
    'source_counts': {
        split: {source: int(((manifest_df['split'] == split) &
                             (manifest_df['source'] == source)).sum())
                for source in sorted(ALLOWED_SOURCES)}
        for split in SPLITS
    },
    'real_writer_counts': {split: len(real_writers[split]) for split in SPLITS},
    'real_writer_disjoint': True,
    'real_writer_held_out_available': REAL_WRITER_HELD_OUT_AVAILABLE,
    'evaluation_domains': {
        str(domain): int(count)
        for domain, count in manifest_df['evaluation_domain'].value_counts().items()
    },
    'evaluation_conditions': {
        str(condition): int(count)
        for condition, count in manifest_df['evaluation_condition'].value_counts().items()
    },
    'synthetic_evaluation_policy': evaluation_policy,
    'synthetic_annotation_sidecar': EVALUATION_ANNOTATIONS_CSV.name,
    'synthetic_annotation_columns': list(EVALUATION_ANNOTATION_COLUMNS),
}
RUN_CONFIGURATION = {
    'profile': PROFILE,
    'hyperparameters': cfg,
    'training_contract': TRAINING_CONTRACT,
    'seed': RUN_SEED,
    'num_workers': NUM_WORKERS,
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'preprocessing_id': EXPECTED_PREPROCESSING_ID,
    'single_gpu': True,
    'deterministic_algorithms': True,
    'dependencies': DEPENDENCY_VERSIONS,
}
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]{0,63}', RUN_TAG):
    raise ValueError('RUN_TAG must be 1-64 portable characters: letters, numbers, dot, underscore, or dash.')
RUN_IDENTITY = {
    'configuration': RUN_CONFIGURATION,
    'dataset_provenance': DATASET_PROVENANCE,
    'split_definition': SPLIT_DEFINITION,
}
RUN_FINGERPRINT = sha256_json(RUN_IDENTITY)
RUN_KEY = f'{RUN_TAG}-{RUN_FINGERPRINT[:12]}'
RUN_DIR = RUNS_ROOT / RUN_KEY
RUN_CONTRACT_PATH = RUN_DIR / 'run-contract.json'
CHECKPOINT_PATH = RUN_DIR / 'training-state.pt'
PROCESSOR_CHECKPOINT_DIR = RUN_DIR / 'checkpoint-processor'
BEST_MODELS_DIR = RUN_DIR / 'best-models'
BASELINE_VALIDATION_RESULT_PATH = RUN_DIR / 'baseline-validation-result.json'
LOCKED_TEST_RESULT_PATH = RUN_DIR / 'locked-test-result.json'
ARCHIVE_PATH = RUNS_ROOT / f'{RUN_KEY}-evaluated.zip'
RUN_CONTRACT = {
    'schema_version': 1,
    'run_key': RUN_KEY,
    'run_fingerprint': RUN_FINGERPRINT,
    **RUN_IDENTITY,
}
RUN_DIR.mkdir(parents=True, exist_ok=True)
if RUN_CONTRACT_PATH.is_file():
    with RUN_CONTRACT_PATH.open('r', encoding='utf-8') as source:
        existing_contract = json.load(source)
    if canonical_json(existing_contract) != canonical_json(RUN_CONTRACT):
        raise RuntimeError('Existing run directory has an incompatible run contract; change RUN_TAG.')
elif any(RUN_DIR.iterdir()):
    raise RuntimeError('Run directory is nonempty but has no run contract; change RUN_TAG or inspect it manually.')
else:
    atomic_write_json(RUN_CONTRACT_PATH, RUN_CONTRACT)
seed_everything(RUN_SEED)
DATA_LOADER_GENERATOR = torch.Generator()
DATA_LOADER_GENERATOR.manual_seed(RUN_SEED)
print('Profile:', PROFILE, cfg)
print('Run key:', RUN_KEY)
print('Output:', RUN_DIR)
print('Dependencies:', DEPENDENCY_VERSIONS)

In [ ]:
# 5. Dependency-free CER / WER / exact-match and grouped reporting
def _levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return prev[n]

def compute_metrics(references, hypotheses):
    if len(references) != len(hypotheses):
        raise ValueError('Prediction/reference count mismatch.')
    if not references:
        raise ValueError('Cannot evaluate an empty group.')
    total_char_errors = total_chars = total_word_errors = total_words = exact = 0
    for reference, hypothesis in zip(references, hypotheses):
        reference, hypothesis = str(reference), str(hypothesis)
        total_char_errors += _levenshtein(list(reference), list(hypothesis))
        total_chars += len(reference)
        ref_words, hyp_words = reference.split(), hypothesis.split()
        total_word_errors += _levenshtein(ref_words, hyp_words)
        total_words += len(ref_words)
        exact += int(reference == hypothesis)
    total = len(references)
    return {
        'cer': total_char_errors / total_chars if total_chars else 0.0,
        'wer': total_word_errors / total_words if total_words else 0.0,
        'accuracy': exact / total, 'exact': exact, 'total': total,
    }

def print_metrics(metrics, title='EVALUATION METRICS'):
    print('=' * 72)
    print(title)
    print('=' * 72)
    print(f"  Samples evaluated : {metrics['total']}")
    print(f"  Exact match       : {metrics['exact']}/{metrics['total']} "
          f"({metrics['accuracy']*100:.2f}%)")
    print(f"  CER               : {metrics['cer']*100:.2f}%")
    print(f"  WER               : {metrics['wer']*100:.2f}%")
    print('=' * 72)

def _metrics_for_frame(frame):
    return compute_metrics(frame['reference'].tolist(), frame['prediction'].tolist())

def _group_metrics(frame, column, explode=False):
    working = frame.explode(column) if explode else frame.copy()
    working[column] = working[column].fillna('').astype(str)
    result = {}
    for value, group in working.groupby(column, sort=True):
        result[str(value or 'unspecified')] = _metrics_for_frame(group)
    return result

def metric_breakdowns(predictions):
    return {
        'overall': _metrics_for_frame(predictions),
        'by_evaluation_domain': _group_metrics(predictions, 'evaluation_domain'),
        'by_evaluation_condition': _group_metrics(
            predictions, 'evaluation_condition'),
        'by_source': _group_metrics(predictions, 'source'),
        'by_field_type': _group_metrics(predictions, 'field_type'),
        'by_format_profile': _group_metrics(predictions, 'format_profile'),
        'by_format_id': _group_metrics(predictions, 'format_id'),
        'by_seen_unseen': _group_metrics(predictions, 'seen_unseen'),
        'by_held_out_condition': _group_metrics(
            predictions, 'held_out_conditions', explode=True),
    }

def print_domain_metrics(breakdowns):
    print('Domain breakdown (never use locked-test groups for model selection):')
    for name, metrics in breakdowns['by_evaluation_domain'].items():
        print(f"  {name:<34} n={metrics['total']:<6} "
              f"CER={metrics['cer']*100:7.2f}%  exact={metrics['accuracy']*100:7.2f}%")
print('[OK] metrics')

In [ ]:
# 6. Load the pinned model only after the dataset/report validation gate passed
def processor_descriptor(active_processor):
    descriptor = {
        'processor_class': type(active_processor).__name__,
        'image_processor_class': type(active_processor.image_processor).__name__,
        'image_processor': active_processor.image_processor.to_dict(),
        'tokenizer_class': type(active_processor.tokenizer).__name__,
        'tokenizer_vocab_size': int(len(active_processor.tokenizer)),
        'tokenizer_model_max_length': int(active_processor.tokenizer.model_max_length),
        'special_token_ids': {
            'bos': active_processor.tokenizer.bos_token_id,
            'eos': active_processor.tokenizer.eos_token_id,
            'sep': active_processor.tokenizer.sep_token_id,
            'pad': active_processor.tokenizer.pad_token_id,
            'unk': active_processor.tokenizer.unk_token_id,
        },
    }
    return json.loads(json.dumps(descriptor, default=str))

processor = TrOCRProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
raw_model = VisionEncoderDecoderModel.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, use_safetensors=True)
resolved_revision = getattr(raw_model.config, '_commit_hash', None)
if resolved_revision and resolved_revision != MODEL_REVISION:
    raise RuntimeError(
        f'Resolved model revision {resolved_revision} does not match pin {MODEL_REVISION}.')
raw_model.config.decoder_start_token_id = processor.tokenizer.sep_token_id
raw_model.config.eos_token_id = processor.tokenizer.sep_token_id
raw_model.config.pad_token_id = processor.tokenizer.pad_token_id
raw_model.config.vocab_size = raw_model.config.decoder.vocab_size
raw_model.generation_config.decoder_start_token_id = raw_model.config.decoder_start_token_id
raw_model.generation_config.eos_token_id = raw_model.config.eos_token_id
raw_model.generation_config.pad_token_id = raw_model.config.pad_token_id
raw_model.generation_config.max_length = cfg['MAX_LABEL_LENGTH']
raw_model.to(device)
PROCESSOR_DESCRIPTOR = processor_descriptor(processor)
if PROCESSOR_CHECKPOINT_DIR.is_dir():
    saved_processor = TrOCRProcessor.from_pretrained(PROCESSOR_CHECKPOINT_DIR)
    if processor_descriptor(saved_processor) != PROCESSOR_DESCRIPTOR:
        raise RuntimeError('Saved checkpoint processor differs from the pinned run processor.')
else:
    processor_temporary = PROCESSOR_CHECKPOINT_DIR.with_name(
        PROCESSOR_CHECKPOINT_DIR.name + '.tmp')
    if processor_temporary.exists():
        shutil.rmtree(processor_temporary)
    processor.save_pretrained(processor_temporary)
    os.replace(processor_temporary, PROCESSOR_CHECKPOINT_DIR)
PROCESSOR_ARTIFACT_SHA256 = hash_directory(PROCESSOR_CHECKPOINT_DIR)
print('Decoder start token id:', raw_model.config.decoder_start_token_id)
print('Pinned model revision:', MODEL_REVISION)
print('Params:', f"{sum(p.numel() for p in raw_model.parameters()):,}")

In [ ]:
# 7. One canonical aspect-preserving transform for train/eval/inference
TROCR_IMAGE_SIZE = 384
TROCR_PAD_COLOR = (255, 255, 255)
PREPROCESSING_ID = 'aspect-pad-384-v1'
if PREPROCESSING_ID != EXPECTED_PREPROCESSING_ID:
    raise RuntimeError('Notebook preprocessing identifier drifted after validation.')

def resize_and_pad(image, size=TROCR_IMAGE_SIZE, fill=TROCR_PAD_COLOR):
    image = image.convert('RGB')
    contained = ImageOps.contain(
        image, (size, size), method=Image.Resampling.BICUBIC)
    canvas = Image.new('RGB', (size, size), fill)
    left = (size - contained.width) // 2
    top = (size - contained.height) // 2
    canvas.paste(contained, (left, top))
    return canvas

def load_preprocessed_image(row):
    path = SPLIT_DIRS[str(row['split'])] / str(row['filename'])
    with Image.open(path) as opened:
        return resize_and_pad(opened)

def processor_pixels(active_processor, images):
    # Already 384x384: disabling processor resize prevents a second geometry transform.
    return active_processor(
        images=images, do_resize=False, return_tensors='pt').pixel_values

def audit_token_lengths(frame, tokenizer, maximum_length):
    lengths = []
    violations = []
    for index, label in enumerate(frame['label'].astype(str)):
        token_count = len(tokenizer(
            label.strip(), add_special_tokens=True, truncation=False)['input_ids'])
        lengths.append(token_count)
        if token_count > maximum_length:
            violations.append((index, token_count, label))
    summary = {
        'maximum_allowed': int(maximum_length),
        'maximum_observed': int(max(lengths)),
        'p95': float(np.percentile(lengths, 95)),
        'p99': float(np.percentile(lengths, 99)),
        'audited_rows': int(len(lengths)),
    }
    if violations:
        examples = '; '.join(
            f'row={index} tokens={length} label={label!r}'
            for index, length, label in violations[:5])
        raise ValueError(
            f'{len(violations)} labels exceed MAX_LABEL_LENGTH={maximum_length}; '
            f'training refuses silent truncation. {examples}')
    return summary

def choose_eval_batch_size(free_memory_bytes, split_size):
    free_gib = free_memory_bytes / (1024 ** 3)
    candidate = 16 if free_gib >= 13 else 8 if free_gib >= 9 else 4 if free_gib >= 6 else 1
    return max(1, min(candidate, int(split_size)))

class HandwrittenDataset(Dataset):
    def __init__(self, frame, active_processor, max_len):
        self.frame = frame.reset_index(drop=True)
        self.processor = active_processor
        self.max_len = max_len

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = load_preprocessed_image(row)
        pixel_values = processor_pixels(self.processor, image).squeeze(0)
        labels = self.processor.tokenizer(
            str(row['label']).strip(), padding='max_length',
            max_length=self.max_len, truncation=False,
            return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

print(f'train={len(train_df)}  val={len(val_df)}  locked_test={len(test_df)}')
print('Training sources:', train_df['source'].value_counts().to_dict())
TOKEN_LENGTH_AUDIT = audit_token_lengths(
    manifest_df, processor.tokenizer, cfg['MAX_LABEL_LENGTH'])
print('Token-length audit:', TOKEN_LENGTH_AUDIT)
free_cuda_memory, _ = torch.cuda.mem_get_info(device)
EVAL_BATCH_SIZE = choose_eval_batch_size(
    free_cuda_memory, max(len(val_df), len(test_df)))
print('Initial evaluation batch size:', EVAL_BATCH_SIZE)
train_ds = HandwrittenDataset(train_df, processor, cfg['MAX_LABEL_LENGTH'])
train_loader = DataLoader(
    train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    worker_init_fn=seed_data_loader_worker,
    generator=DATA_LOADER_GENERATOR,
    persistent_workers=(NUM_WORKERS > 0))
if len(train_loader) < 1:
    raise RuntimeError('Training loader is empty.')

In [ ]:
# 8. Optimizer, freezing, exact scheduler sizing, and resumable state
encoder_params = list(raw_model.encoder.parameters())
decoder_params = [p for name, p in raw_model.named_parameters()
                  if not name.startswith('encoder.')]
optimizer_settings = TRAINING_CONTRACT['optimizer']
optimizer = AdamW([
    {'params': decoder_params, 'lr': cfg['DECODER_LR']},
    {'params': encoder_params, 'lr': cfg['ENCODER_LR']},
], weight_decay=optimizer_settings['weight_decay'],
   betas=tuple(optimizer_settings['betas']), eps=optimizer_settings['eps'])
if cfg['FREEZE_ENCODER']:
    for parameter in raw_model.encoder.parameters():
        parameter.requires_grad = False
    print('Encoder frozen until epoch', cfg['UNFREEZE_AT_EPOCH'] + 1)
steps_per_epoch = max(1, math.ceil(len(train_loader) / cfg['GRAD_ACCUM_STEPS']))
total_steps = steps_per_epoch * cfg['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * cfg['WARMUP_RATIO']), total_steps)
model = raw_model
scaler = GradScaler(TRAINING_CONTRACT['amp']['device_type'])

def capture_rng_states():
    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_cuda': torch.cuda.get_rng_state_all(),
        'data_loader_generator': DATA_LOADER_GENERATOR.get_state(),
    }

def restore_rng_states(states):
    random.setstate(states['python'])
    np.random.set_state(states['numpy'])
    torch.set_rng_state(states['torch_cpu'])
    torch.cuda.set_rng_state_all(states['torch_cuda'])
    DATA_LOADER_GENERATOR.set_state(states['data_loader_generator'])

def checkpoint_payload(completed_epoch, best_cer, patience, best_artifact):
    return {
        'checkpoint_schema_version': 1,
        'completed_epoch': int(completed_epoch),
        'best_cer': float(best_cer),
        'patience': int(patience),
        'best_artifact': best_artifact,
        'run_key': RUN_KEY,
        'run_fingerprint': RUN_FINGERPRINT,
        'configuration': RUN_CONFIGURATION,
        'dataset_provenance': DATASET_PROVENANCE,
        'model_state': raw_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'processor_artifact': str(PROCESSOR_CHECKPOINT_DIR),
        'processor_artifact_sha256': PROCESSOR_ARTIFACT_SHA256,
        'rng_states': capture_rng_states(),
    }

def save_training_checkpoint(completed_epoch, best_cer, patience, best_artifact):
    temporary = CHECKPOINT_PATH.with_name(CHECKPOINT_PATH.name + '.tmp')
    torch.save(
        checkpoint_payload(completed_epoch, best_cer, patience, best_artifact),
        temporary)
    os.replace(temporary, CHECKPOINT_PATH)

def validate_checkpoint_contract(state):
    if state.get('checkpoint_schema_version') != 1:
        raise RuntimeError('Checkpoint schema is unsupported.')
    if state.get('run_key') != RUN_KEY or state.get('run_fingerprint') != RUN_FINGERPRINT:
        raise RuntimeError('Checkpoint run key/fingerprint does not match this run.')
    if canonical_json(state.get('configuration')) != canonical_json(RUN_CONFIGURATION):
        raise RuntimeError('Checkpoint configuration does not match this run.')
    if canonical_json(state.get('dataset_provenance')) != canonical_json(DATASET_PROVENANCE):
        raise RuntimeError('Checkpoint dataset provenance does not match this dataset.')
    if state.get('processor_artifact_sha256') != PROCESSOR_ARTIFACT_SHA256:
        raise RuntimeError('Checkpoint processor artifact does not match this run.')

start_epoch = 1
best_cer = float('inf')
patience = 0
best_artifact = None
if CHECKPOINT_PATH.is_file():
    if not RESUME_IF_AVAILABLE:
        raise RuntimeError('Checkpoint exists but RESUME_IF_AVAILABLE=False; change RUN_TAG for a fresh run.')
    print('SECURITY: resume checkpoints are pickle files; load only a trusted run you created.')
    resume_state = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    validate_checkpoint_contract(resume_state)
    raw_model.load_state_dict(resume_state['model_state'])
    optimizer.load_state_dict(resume_state['optimizer_state'])
    scheduler.load_state_dict(resume_state['scheduler_state'])
    scaler.load_state_dict(resume_state['scaler_state'])
    restore_rng_states(resume_state['rng_states'])
    start_epoch = int(resume_state['completed_epoch']) + 1
    best_cer = float(resume_state['best_cer'])
    patience = int(resume_state['patience'])
    best_artifact = resume_state.get('best_artifact')
    del resume_state
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Resumed after epoch {start_epoch - 1}; next epoch is {start_epoch}.')
else:
    print('Starting a fresh training run.')
if (cfg['FREEZE_ENCODER']
        and start_epoch > cfg['UNFREEZE_AT_EPOCH'] + 1):
    for parameter in raw_model.encoder.parameters():
        parameter.requires_grad = True
    print('Restored post-unfreeze encoder state for resumed training.')
print('[OK] optimizer')

In [ ]:
# 9. Shared evaluator/inference path (uses the same resize-and-pad transform)
@torch.no_grad()
def _predict_dataframe_once(frame, active_model, active_processor, batch_size):
    if frame.empty:
        raise ValueError('Cannot evaluate an empty split.')
    active_model.eval()
    predictions = []
    for start_index in range(0, len(frame), batch_size):
        chunk = frame.iloc[start_index:start_index + batch_size]
        images = [load_preprocessed_image(row) for _, row in chunk.iterrows()]
        pixel_values = processor_pixels(active_processor, images).to(device)
        generated = active_model.generate(
            pixel_values, max_length=cfg['MAX_LABEL_LENGTH'])
        decoded = active_processor.batch_decode(generated, skip_special_tokens=True)
        predictions.extend(output.strip() for output in decoded)
    if len(predictions) != len(frame):
        raise RuntimeError('Prediction count does not match evaluated rows.')
    result = frame.copy().reset_index(drop=True)
    result['reference'] = result['label'].astype(str).str.strip()
    result['prediction'] = predictions
    return result

def predict_dataframe(frame, eval_model=None, eval_processor=None,
                      batch_size=None):
    if frame.empty:
        raise ValueError('Cannot evaluate an empty split.')
    active_model = eval_model if eval_model is not None else raw_model
    active_processor = eval_processor if eval_processor is not None else processor
    attempted_batch_size = min(int(batch_size or EVAL_BATCH_SIZE), len(frame))
    while True:
        try:
            return _predict_dataframe_once(
                frame, active_model, active_processor, attempted_batch_size)
        except torch.cuda.OutOfMemoryError:
            if attempted_batch_size == 1:
                raise
            attempted_batch_size = max(1, attempted_batch_size // 2)
            torch.cuda.empty_cache()
            print('Evaluation OOM; retrying with batch size', attempted_batch_size)

def evaluate_dataframe(frame, eval_model=None, eval_processor=None,
                       show=False, title='METRICS'):
    predictions = predict_dataframe(
        frame, eval_model=eval_model, eval_processor=eval_processor)
    breakdowns = metric_breakdowns(predictions)
    if show:
        print_metrics(breakdowns['overall'], title)
        print_domain_metrics(breakdowns)
    return breakdowns['overall'], predictions, breakdowns
print('[OK] evaluator')

In [ ]:
# 9b. Base-model comparison uses VALIDATION, never the locked test
if BASELINE_VALIDATION_RESULT_PATH.is_file():
    with BASELINE_VALIDATION_RESULT_PATH.open('r', encoding='utf-8') as source:
        baseline_cache = json.load(source)
    if baseline_cache.get('manifest_sha256') != manifest_sha256:
        raise RuntimeError('Baseline cache belongs to a different manifest.')
    if baseline_cache.get('images_sha256') != images_sha256:
        raise RuntimeError('Baseline cache belongs to different image content.')
    if baseline_cache.get('model_revision') != MODEL_REVISION:
        raise RuntimeError('Baseline cache uses a different base-model revision.')
    base_val_predictions = pd.DataFrame(baseline_cache.get('predictions', []))
    expected_pairs = list(zip(val_df['filename'], val_df['label']))
    cached_pairs = list(zip(base_val_predictions.get('filename', []),
                            base_val_predictions.get('label', [])))
    if cached_pairs != expected_pairs:
        raise RuntimeError('Baseline cache rows do not match validation.')
    base_val_predictions['held_out_conditions'] = base_val_predictions[
        'held_out_conditions'].apply(tuple)
    base_val_breakdowns = metric_breakdowns(base_val_predictions)
    base_val_metrics = base_val_breakdowns['overall']
    print_metrics(base_val_metrics, 'BASE MODEL — CACHED DEVELOPMENT VALIDATION')
else:
    if CHECKPOINT_PATH.is_file():
        raise RuntimeError(
            'Training checkpoint exists but baseline cache is missing; raw_model is no longer the base weights.')
    print('Evaluating the pinned base model on validation data...')
    base_val_metrics, base_val_predictions, base_val_breakdowns = evaluate_dataframe(
        val_df, show=True, title='BASE MODEL — DEVELOPMENT VALIDATION')
    atomic_write_json(BASELINE_VALIDATION_RESULT_PATH, {
        'schema_version': 1,
        'manifest_sha256': manifest_sha256,
        'images_sha256': images_sha256,
        'model_name': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'predictions': base_val_predictions.to_dict('records'),
    })

In [ ]:
# 10. Training loop — all checkpoint/early-stop decisions use validation
BEST_MODELS_DIR.mkdir(parents=True, exist_ok=True)
def save_best_artifact(epoch, validation_cer):
    artifact_name = f'epoch-{epoch:03d}-valcer-{validation_cer:.8f}'
    destination = BEST_MODELS_DIR / artifact_name
    temporary = BEST_MODELS_DIR / (artifact_name + '.tmp')
    if destination.exists():
        return artifact_name
    if temporary.exists():
        shutil.rmtree(temporary)
    temporary.mkdir(parents=True)
    raw_model.save_pretrained(temporary, safe_serialization=True)
    processor.save_pretrained(temporary)
    os.replace(temporary, destination)
    return artifact_name

def should_skip_training(start_epoch, epochs, patience, patience_threshold):
    return start_epoch > epochs or patience >= patience_threshold

training_already_complete = should_skip_training(
    start_epoch, cfg['EPOCHS'], patience, cfg['EARLY_STOP_PATIENCE'])
if training_already_complete:
    if start_epoch > cfg['EPOCHS']:
        print('Checkpoint already completed the configured epochs; skipping training.')
    else:
        print('Checkpoint already reached early stopping; skipping further training.')
started = time.time()
training_epochs = () if training_already_complete else range(
    start_epoch, cfg['EPOCHS'] + 1)
for epoch in training_epochs:
    if cfg['FREEZE_ENCODER'] and epoch == cfg['UNFREEZE_AT_EPOCH'] + 1:
        for parameter in raw_model.encoder.parameters():
            parameter.requires_grad = True
        print(f'  -> Encoder unfrozen at epoch {epoch}')
    model.train()
    running = 0.0
    optimizer.zero_grad()
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg["EPOCHS"]}')
    for step, batch in enumerate(progress, start=1):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        group_start = ((step - 1) // cfg['GRAD_ACCUM_STEPS']) * cfg['GRAD_ACCUM_STEPS']
        group_start_sample = group_start * cfg['BATCH_SIZE']
        accumulation_group_samples = min(
            cfg['GRAD_ACCUM_STEPS'] * cfg['BATCH_SIZE'],
            len(train_ds) - group_start_sample)
        with autocast(
                TRAINING_CONTRACT['amp']['device_type'], dtype=torch.float16):
            output = model(pixel_values=pixel_values, labels=labels)
            raw_loss = output.loss.mean()
            loss = raw_loss * labels.shape[0] / accumulation_group_samples
        scaler.scale(loss).backward()
        running += raw_loss.item()
        should_step = (
            step % cfg['GRAD_ACCUM_STEPS'] == 0 or step == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), TRAINING_CONTRACT['gradient_clip_max_norm'])
            scale_before_step = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scaler.get_scale() >= scale_before_step:
                scheduler.step()
        progress.set_postfix(
            loss=f'{running/step:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
    val_metrics, _, _ = evaluate_dataframe(val_df)
    print(f"  Epoch {epoch}: train_loss={running/len(train_loader):.4f}  "
          f"val_CER={val_metrics['cer']*100:.2f}%  "
          f"val_acc={val_metrics['accuracy']*100:.2f}%")
    if val_metrics['cer'] < best_cer:
        best_cer = val_metrics['cer']
        best_artifact = save_best_artifact(epoch, best_cer)
        patience = 0
        print(f'    saved best model (val_CER={best_cer*100:.2f}%)')
    else:
        patience += 1
        print(f'    no improvement ({patience}/{cfg["EARLY_STOP_PATIENCE"]})')
    save_training_checkpoint(epoch, best_cer, patience, best_artifact)
    if patience >= cfg['EARLY_STOP_PATIENCE']:
        print('    early stopping')
        break
if not best_artifact or not (BEST_MODELS_DIR / best_artifact).is_dir():
    raise RuntimeError('No compatible best-model artifact exists for this run.')
print(f'\nDone in {(time.time()-started)/60:.1f} min. '
      f'Best validation CER: {best_cer*100:.2f}%')

In [ ]:
# 11. Freeze choices, reload the best artifact, then touch locked TEST once
BEST_ARTIFACT_DIR = BEST_MODELS_DIR / best_artifact
best_processor = TrOCRProcessor.from_pretrained(BEST_ARTIFACT_DIR)
best_model = VisionEncoderDecoderModel.from_pretrained(BEST_ARTIFACT_DIR).to(device)
if processor_descriptor(best_processor) != PROCESSOR_DESCRIPTOR:
    raise RuntimeError('Best artifact processor differs from the evaluated run contract.')
best_model.eval()
EVALUATED_ARTIFACT_SHA256 = hash_directory(BEST_ARTIFACT_DIR)
best_val_metrics, best_val_predictions, best_val_breakdowns = evaluate_dataframe(
    val_df, eval_model=best_model, eval_processor=best_processor, show=True,
    title='BEST FINE-TUNED MODEL — DEVELOPMENT VALIDATION')

# Set True only after profile/hyperparameters/checkpoint choice are frozen.
MODEL_CHOICES_FROZEN = False
if globals().get('_LOCKED_TEST_EVALUATED', False):
    raise RuntimeError(
        'Locked test already evaluated in this kernel. Do not rerun this cell.')
if not MODEL_CHOICES_FROZEN:
    raise RuntimeError('Freeze all model choices before final test evaluation.')
if LOCKED_TEST_RESULT_PATH.is_file():
    with LOCKED_TEST_RESULT_PATH.open('r', encoding='utf-8') as source:
        locked_cache = json.load(source)
    if locked_cache.get('run_fingerprint') != RUN_FINGERPRINT:
        raise RuntimeError('Locked-test cache belongs to a different run fingerprint.')
    if locked_cache.get('evaluated_artifact_sha256') != EVALUATED_ARTIFACT_SHA256:
        raise RuntimeError('Locked-test cache belongs to different evaluated weights.')
    test_predictions = pd.DataFrame(locked_cache.get('predictions', []))
    expected_pairs = list(zip(test_df['filename'], test_df['label']))
    cached_pairs = list(zip(test_predictions.get('filename', []),
                            test_predictions.get('label', [])))
    if cached_pairs != expected_pairs:
        raise RuntimeError('Locked-test cache rows do not match the current test split.')
    test_predictions['held_out_conditions'] = test_predictions[
        'held_out_conditions'].apply(tuple)
    test_breakdowns = metric_breakdowns(test_predictions)
    test_metrics = test_breakdowns['overall']
    print_metrics(test_metrics, 'LOCKED FINAL TEST — CACHED ONE-SHOT RESULT')
    print_domain_metrics(test_breakdowns)
    print('Loaded locked-test predictions; test images were not evaluated again.')
else:
    test_metrics, test_predictions, test_breakdowns = evaluate_dataframe(
        test_df, eval_model=best_model, eval_processor=best_processor, show=True,
        title='LOCKED FINAL TEST — OVERALL (ONE-SHOT)')
    atomic_write_json(LOCKED_TEST_RESULT_PATH, {
        'schema_version': 1,
        'run_fingerprint': RUN_FINGERPRINT,
        'evaluated_artifact_sha256': EVALUATED_ARTIFACT_SHA256,
        'predictions': test_predictions.to_dict('records'),
    })
_LOCKED_TEST_EVALUATED = True
print('Synthetic test naming: synthetic / in-distribution or synthetic / held-out')
print('Real test naming     : real / writer-held-out')
print('Evaluated model artifact:', BEST_ARTIFACT_DIR)
print('Evaluated artifact SHA-256:', EVALUATED_ARTIFACT_SHA256)

In [ ]:
# 12. Export one artifact-bound report from the cached one-shot predictions
if not globals().get('_LOCKED_TEST_EVALUATED', False):
    raise RuntimeError('Run the locked final-test cell exactly once first.')
weights_file = next((name for name in ('model.safetensors', 'pytorch_model.bin')
                     if (BEST_ARTIFACT_DIR / name).is_file()), None)
if weights_file is None:
    raise FileNotFoundError('Saved model weights were not found.')
report = {
    'schema_version': 3,
    'run_key': RUN_KEY,
    'run_fingerprint': RUN_FINGERPRINT,
    'model_key': best_artifact,
    'base_model': {'name': MODEL_NAME, 'revision': MODEL_REVISION},
    'dataset': DATA_ROOT.name,
    'manifest_schema_version': int(MANIFEST_SCHEMA_VERSION),
    'manifest_sha256': manifest_sha256,
    'images_sha256': images_sha256,
    'dataset_validation_sha256': sha256_file(VALIDATION_JSON),
    'dataset_provenance': DATASET_PROVENANCE,
    'split_definition': SPLIT_DEFINITION,
    'generator_provenance': {
        'run_metadata_sha256': sha256_file(RUN_METADATA_JSON),
        'generator_revision': generator_run_metadata.get('generator_revision'),
        'generator_seed': generator_run_metadata.get('seed'),
        'preprocessing': generator_preprocessing,
        'evaluation_policy': evaluation_policy,
        'evaluation_annotations_file': EVALUATION_ANNOTATIONS_CSV.name,
        'evaluation_annotations_sha256': sha256_file(
            EVALUATION_ANNOTATIONS_CSV),
    },
    'profile': PROFILE,
    'hyperparameters': cfg,
    'seeds': {
        'run': RUN_SEED, 'python': RUN_SEED, 'numpy': RUN_SEED,
        'torch_cpu': RUN_SEED, 'torch_cuda': RUN_SEED,
        'data_loader_generator': RUN_SEED,
        'workers': 'torch.initial_seed modulo 2**32',
    },
    'dependencies': DEPENDENCY_VERSIONS,
    'determinism_guarantee': DETERMINISM_GUARANTEE,
    'processor': PROCESSOR_DESCRIPTOR,
    'processor_artifact_sha256': PROCESSOR_ARTIFACT_SHA256,
    'token_length_audit': TOKEN_LENGTH_AUDIT,
    'preprocessing': {
        'id': PREPROCESSING_ID, 'size': TROCR_IMAGE_SIZE,
        'pad_color_rgb': list(TROCR_PAD_COLOR), 'resample': 'bicubic',
        'preserve_aspect_ratio': True,
    },
    'development_validation': {
        'base_model': base_val_metrics, 'fine_tuned_model': best_val_metrics,
        'base_predictions_sha256': sha256_file(
            BASELINE_VALIDATION_RESULT_PATH),
    },
    'locked_final_test': {
        'split': 'test', 'one_shot': True,
        'sample_count': int(test_metrics['total']),
        'metrics': test_metrics, 'breakdowns': test_breakdowns,
        'domain_names': {
            'synthetic_in_distribution': 'synthetic / in-distribution',
            'synthetic_held_out': 'synthetic / held-out',
            'real_writer_held_out': 'real / writer-held-out',
        },
    },
    'evaluated_at': datetime.now(timezone.utc).isoformat(),
    'locked_test_predictions_sha256': sha256_file(LOCKED_TEST_RESULT_PATH),
    'evaluated_artifact': best_artifact,
    'evaluated_artifact_sha256': EVALUATED_ARTIFACT_SHA256,
    'weights_file': weights_file,
    'weights_sha256': sha256_file(BEST_ARTIFACT_DIR / weights_file),
}
report_path = RUN_DIR / 'evaluation-report.json'
atomic_write_json(report_path, report)
print('Evaluation report:', report_path)
print(json.dumps(report, indent=2))

In [ ]:
# 13. Side-by-side DEVELOPMENT VALIDATION comparison (never test)
def _percent(value):
    return f'{value*100:.2f}%'
print('=' * 68)
print('BASE vs FINE-TUNED — DEVELOPMENT VALIDATION')
print(f"{'Metric':<12}{'Base':>13}{'Fine-tuned':>15}{'Verdict':>14}")
print('-' * 68)
for key, name in [('cer', 'CER'), ('wer', 'WER'), ('accuracy', 'Accuracy')]:
    base_value, fine_tuned_value = base_val_metrics[key], best_val_metrics[key]
    if key in ('cer', 'wer'):
        verdict = ('better' if fine_tuned_value < base_value
                   else 'same' if fine_tuned_value == base_value else 'worse')
    else:
        verdict = ('better' if fine_tuned_value > base_value
                   else 'same' if fine_tuned_value == base_value else 'worse')
    print(f'{name:<12}{_percent(base_value):>13}'
          f'{_percent(fine_tuned_value):>15}{verdict:>14}')
print('=' * 68)
print('Validation samples:', base_val_metrics['total'])
print('Locked test metrics above are final reporting only, never a comparison input.')

In [ ]:
# 14. Package the evaluated checkpoint and mandatory report
report_path = RUN_DIR / 'evaluation-report.json'
if not report_path.is_file():
    raise FileNotFoundError('evaluation-report.json is required before packaging.')
with report_path.open('r', encoding='utf-8') as source:
    packaged_report = json.load(source)
current_artifact_sha256 = hash_directory(BEST_ARTIFACT_DIR)
if current_artifact_sha256 != EVALUATED_ARTIFACT_SHA256:
    raise RuntimeError('Best-model artifact changed after evaluation; refusing to package.')
if packaged_report.get('evaluated_artifact_sha256') != current_artifact_sha256:
    raise RuntimeError('Report does not identify the exact evaluated model artifact.')
if packaged_report.get('weights_sha256') != sha256_file(BEST_ARTIFACT_DIR / weights_file):
    raise RuntimeError('Report weight hash does not match the evaluated weights.')
if ARCHIVE_PATH.exists():
    raise FileExistsError(
        f'Archive already exists and will not be overwritten: {ARCHIVE_PATH}')
temporary_archive = ARCHIVE_PATH.with_name(ARCHIVE_PATH.name + '.tmp')
with zipfile.ZipFile(temporary_archive, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(BEST_ARTIFACT_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, Path('model') / path.relative_to(BEST_ARTIFACT_DIR))
    archive.write(report_path, 'evaluation-report.json')
    archive.write(RUN_CONTRACT_PATH, 'run-contract.json')
    archive.write(
        BASELINE_VALIDATION_RESULT_PATH, 'baseline-validation-result.json')
    archive.write(LOCKED_TEST_RESULT_PATH, 'locked-test-result.json')
    archive.write(MANIFEST_CSV, 'dataset-provenance/manifest.csv')
    archive.write(VALIDATION_JSON, 'dataset-provenance/dataset-validation.json')
    archive.write(RUN_METADATA_JSON, 'dataset-provenance/run-metadata.json')
    archive.write(
        EVALUATION_ANNOTATIONS_CSV,
        'dataset-provenance/evaluation-annotations.csv')
    archive.writestr('artifact-integrity.json', canonical_json({
        'schema_version': 1,
        'run_key': RUN_KEY,
        'evaluated_artifact_sha256': current_artifact_sha256,
        'weights_file': f'model/{weights_file}',
        'weights_sha256': packaged_report['weights_sha256'],
        'evaluation_report_sha256': sha256_file(report_path),
        'run_contract_sha256': sha256_file(RUN_CONTRACT_PATH),
        'baseline_validation_result_sha256': sha256_file(
            BASELINE_VALIDATION_RESULT_PATH),
        'locked_test_result_sha256': sha256_file(LOCKED_TEST_RESULT_PATH),
        'manifest_sha256': manifest_sha256,
        'images_sha256': images_sha256,
        'dataset_validation_sha256': sha256_file(VALIDATION_JSON),
        'generator_run_metadata_sha256': sha256_file(RUN_METADATA_JSON),
        'evaluation_annotations_sha256': sha256_file(
            EVALUATION_ANNOTATIONS_CSV),
    }))
with zipfile.ZipFile(temporary_archive, 'r') as archive:
    if archive.testzip() is not None:
        raise RuntimeError('Packaged ZIP failed its CRC verification.')
    required = {
        'evaluation-report.json', 'run-contract.json', 'artifact-integrity.json',
        'baseline-validation-result.json', 'locked-test-result.json',
        'dataset-provenance/manifest.csv',
        'dataset-provenance/dataset-validation.json',
        'dataset-provenance/run-metadata.json',
        'dataset-provenance/evaluation-annotations.csv',
        f'model/{weights_file}', 'model/config.json',
        'model/preprocessor_config.json',
    }
    missing = required - set(archive.namelist())
    if missing:
        raise RuntimeError(f'Packaged ZIP lacks required files: {sorted(missing)}')
os.replace(temporary_archive, ARCHIVE_PATH)
print('zipped ->', ARCHIVE_PATH)
print('archive SHA-256 ->', sha256_file(ARCHIVE_PATH))